## Ingest pitstop.json

In [0]:
%run /Workspace/Users/nicolas97alonso@gmail.com/databricks-course/utils/vairables

### Step 1 - Ingest pit_stops.json

#### 1.1 Define the schema

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

In [0]:
pitstops_schema = StructType([
    StructField("driverId", StringType(), False),
    StructField("duration", FloatType(), True),
    StructField("lap", IntegerType(), True),
    StructField("milliseconds", IntegerType(), True),
    StructField("raceId", IntegerType(), True),
    StructField("stop", IntegerType(), True),
    StructField("time", StringType(), True)
])

#### 1.2 Read json file

In [0]:
raw_path = define_path("raw")

In [0]:
pitstops_df = (spark.read
               .format("json")
               .schema(pitstops_schema)
               .option("multiLine", True)
               .load(f"{raw_path}/pit_stops.json")
               )

### Step 2 - Rename Columns & Add columns

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
renamed_pistops_df = (
    pitstops_df
        .withColumnRenamed("raceId", "race_id")
        .withColumnRenamed("driverId", "driver_id")
        .withColumn("ingestion_date", current_timestamp())
)

### Step 3 - Write parquet file 

In [0]:
processed_path = define_path("processed")

In [0]:
renamed_pistops_df.write.format("parquet").mode("overwrite").save(f"{processed_path}/pit_stops")